# Bank 数据集 Governance 实验

本 notebook 用于运行第二篇 governance 框架在 Bank 数据集上的实验。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("当前项目目录：", PROJECT_ROOT)


当前项目目录： e:\yan\组\三支决策\机器学习\BT_TWD_v2


In [2]:
CONFIG_PATH = "configs/bank_bttwd.yaml"

# 是否运行消融实验：
# True：运行 no_cp 和 no_progressive 两个消融；False：只运行 full，跳过消融单元。
RUN_ABLATION = False

print("当前配置文件：", CONFIG_PATH)
print("运行消融实验：", RUN_ABLATION)


当前配置文件： configs/bank_bttwd.yaml
运行消融实验： False


## 结果摘要工具函数


In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

output_root = Path("outputs/governance")

def show_governance_summary(mode: str) -> None:
    """Read and display the current dataset summary for one run mode."""
    config_stem = Path(CONFIG_PATH).stem
    print(f"\n===== {mode} results =====")

    dataset_summary = output_root / mode / "dataset_summary.csv"
    fold_summary = output_root / mode / config_stem / "fold_summary.csv"
    sample_records = output_root / mode / config_stem / "sample_records.csv"

    if dataset_summary.exists():
        print("Reading:", dataset_summary)
        summary_df = pd.read_csv(dataset_summary)
        if "dataset_name" in summary_df.columns:
            dataset_row = summary_df[summary_df["dataset_name"].astype(str) == config_stem]
            if dataset_row.empty:
                available = ", ".join(summary_df["dataset_name"].astype(str).tolist())
                print(f"No row for {config_stem}; available datasets: {available}")
            else:
                display(dataset_row.reset_index(drop=True))
        else:
            print("dataset_summary.csv has no dataset_name column; cannot filter by dataset.")
    else:
        print("Missing dataset_summary.csv:", dataset_summary)

    if fold_summary.exists():
        print("Reading:", fold_summary)
        display(pd.read_csv(fold_summary))
    else:
        print("Missing fold_summary.csv:", fold_summary)

    if sample_records.exists():
        print("Sample records:", sample_records)
    else:
        print("Missing sample_records.csv:", sample_records)


## 运行完整 governance 实验


In [4]:
import subprocess

cmd = [
    sys.executable,
    "scripts/run_governance_experiments.py",
    "--config",
    CONFIG_PATH,
]

print("运行命令：", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

print("标准输出：")
print(result.stdout)

print("错误输出：")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"实验运行失败，返回码：{result.returncode}")

show_governance_summary("full")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/bank_bttwd.yaml
标准输出：
【INFO】【2026-05-18 19:22:19】【数据加载】文本表格 E:\yan\组\三支决策\机器学习\BT_TWD_v2\data\bank\bank-full.csv 已读取，样本数=45211，列数=17
【INFO】【2026-05-18 19:22:19】【数据加载】标签列 y 已处理完成：dropna_target=False, 丢弃样本=0, 最终样本数=45211, 正类比例=11.70%
【INFO】【2026-05-18 19:22:19】【数据加载】银行营销数据集已读取，标签已映射为0/1，样本数=45211，正类比例=11.70%
【INFO】【2026-05-18 19:22:19】【数据集信息】名称=bank_full，样本数=45211，目标列=y，正类比例=11.70%
【INFO】【2026-05-18 19:22:19】【预处理】连续特征=7个，类别特征=9个
【INFO】【2026-05-18 19:22:19】【预处理】编码后维度=42
【INFO】【2026-05-18 19:22:19】【governance】bank_bttwd 第 1/5 折
【INFO】【2026-05-18 19:22:20】【桶树】已为样本生成桶ID，共 86 个组合
【INFO】【2026-05-18 19:22:20】【桶树】已为样本生成桶ID，共 80 个组合
【INFO】【2026-05-18 19:22:37】【governance】weak bucket resolver：strong=22，weak=101
【INFO】【2026-05-18 19:22:37】【桶树】已为样本生成桶ID，共 82 个组合
【INFO】【2026-05-18 19:23:01】【governance】bank_bttwd 第 2/5 折
【INFO】【2026-05-18 19:23:02】【桶树】已为样本生成桶ID，共 84 个组合
【INFO】【2026-05-18 19:23:02】【桶树】已为样本生成桶ID，共 82 个组

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,bank_bttwd,0.116089,0.820462,0.62757,0.900621,0.11714,0.80603,0.612243,0.898454,1.0,...,0.857143,0.285714,0.714286,0.142857,0.761905,0.333333,0.428571,0.571429,7.0,2.0


Reading: outputs\governance\full\bank_bttwd\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,bank_bttwd,1,0.112463,0.826158,0.637838,0.903682,0.114315,0.824292,0.624544,0.897490,...,1.0,0.166667,0.833333,0.0,0.875000,0.250000,0.625000,0.833333,3.75,0.0
1,bank_bttwd,2,0.120908,0.809256,0.615126,0.898695,0.118503,0.791921,0.602010,0.899248,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00,0.0
2,bank_bttwd,3,0.119332,0.817004,0.618033,0.896925,0.121433,0.809488,0.602595,0.891617,...,1.0,0.500000,0.500000,0.0,0.666667,0.333333,0.333333,0.500000,1.50,0.0
3,bank_bttwd,4,0.110457,0.831956,0.642270,0.903782,0.112586,0.804391,0.622436,0.904335,...,1.0,0.000000,1.000000,0.0,1.000000,0.333333,0.666667,1.000000,1.50,0.0
4,bank_bttwd,5,0.117286,0.817938,0.624585,0.900022,0.118862,0.800059,0.609630,0.899580,...,0.5,0.500000,0.500000,0.5,0.500000,0.500000,0.000000,0.000000,0.75,2.0


Sample records: outputs\governance\full\bank_bttwd\sample_records.csv


## 运行无 CP 消融


In [5]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-cp-validation",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无 CP 消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_cp")
else:
    print("RUN_ABLATION=False，跳过无 CP 消融。")


RUN_ABLATION=False，跳过无 CP 消融。


## 运行无渐进更新消融


In [6]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-progressive-update",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无渐进更新消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_progressive")
else:
    print("RUN_ABLATION=False，跳过无渐进更新消融。")


RUN_ABLATION=False，跳过无渐进更新消融。
